In [1]:
import torch
import numpy as np
from factor_analyzer.rotator import Rotator

# 1) Load parameters
model_data = torch.load('./output/mirt_model_k6.pt', map_location=torch.device('cpu'))
theta = model_data['theta']          # shape: (n_people, k)
a = model_data['a']                  # shape: (n_items,  k)
b = model_data['b']

print("--- Initial Loaded Data ---")
print(f"Original theta shape: {theta.shape}")
print(f"Original 'a' matrix shape: {a.shape}\n")

a_numpy = a.detach().cpu().numpy()
theta_numpy = theta.detach().cpu().numpy()

# 2) Rotate 'a'
rotator = Rotator(method='varimax')  # orthogonal
a_rotated_numpy = rotator.fit_transform(a_numpy)
print("--- After Rotation of 'a' ---")
print(f"Rotated 'a' matrix shape: {a_rotated_numpy.shape}\n")

# 3) Transform theta using the SAME rotation matrix from Rotator
T = rotator.rotation_                # <- correct attribute on Rotator
theta_transformed_numpy = theta_numpy @ T  # for orthogonal varimax, use R (not inv(R))

print("--- After Transformation of 'theta' ---")
print(f"Transformed theta shape: {theta_transformed_numpy.shape}\n")

# (Optional) Sanity check: inner products preserved (up to numerical tolerance)
# err = np.max(np.abs(a_numpy @ theta_numpy.T - a_rotated_numpy @ theta_transformed_numpy.T))
# print(f"Max abs diff in item-person inner products: {err:.3e}")

# 4) Standardize transformed theta
# Z-score theta for interpretability
m = theta_transformed_numpy.mean(axis=0)           # shape (K,)
s = theta_transformed_numpy.std(axis=0)            # shape (K,)
eps = 1e-8
D = np.diag(np.maximum(s, eps))
theta_z_scores_numpy = (theta_transformed_numpy - m) / np.maximum(s, eps)

# If you intend to use theta_z for prediction, adjust a and b:
a_for_z_numpy = a_rotated_numpy @ D                # (items x K)
b_for_z_numpy = b.detach().cpu().numpy() - a_rotated_numpy @ m  # (items,)

print("--- Final Standardized Z-Scores (from transformed theta) ---")
print("These are the scores you should use for interpretation.")
print(theta_z_scores_numpy)

--- Initial Loaded Data ---
Original theta shape: torch.Size([183, 6])
Original 'a' matrix shape: torch.Size([78712, 6])

--- After Rotation of 'a' ---
Rotated 'a' matrix shape: (78712, 6)

--- After Transformation of 'theta' ---
Transformed theta shape: (183, 6)

--- Final Standardized Z-Scores (from transformed theta) ---
These are the scores you should use for interpretation.
[[ 1.5905465   3.49991235 -2.70676842  0.80146618  0.72562385 -0.62409166]
 [ 1.26710786 -0.50314049  1.10198573 -1.1741239   0.34140497  0.07110775]
 [ 1.86002922  0.60271614  0.90773613  0.07469808 -0.76376825 -1.55688678]
 ...
 [-1.05323766 -0.53844968 -2.76657604  1.42263159  0.66665258  2.97106778]
 [-0.43951293 -0.54343183 -2.60149674  2.23806708  0.32196824  3.12436554]
 [ 0.04734164 -0.47165056 -1.30415358  1.18832103  1.00318341  2.02381296]]


In [2]:
import pandas as pd

resmat = pd.read_pickle("../data/resmat.pkl")

# Step 1: Get the final theta tensor into a NumPy array
theta_abilities = theta_z_scores_numpy
# Step 2: Create a labeled pandas DataFrame
# Use the model names from your original resmat for the index
model_names = resmat.index
factor_names = [f'F{i+1}' for i in range(theta.shape[1])]
ability_df = pd.DataFrame(theta_abilities, index=model_names, columns=factor_names)

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
pd.set_option('display.max_rows', None)

# --- Assume you have these DataFrames ---
df_theta = ability_df
df_performance = pd.read_csv('../data/scenario_probs.csv')  # (models x scenarios) DataFrame of raw scores

# Step 1: Create the "Ground Truth" Score (your target variable 'y')
scenarios = ['med_qa']
ground_truth_scores = df_performance[scenarios].mean(axis=1)

# The factor scores are your predictor variables 'X'
X = df_theta 
y = ground_truth_scores

# Step 2 & 3: Run the regression
reg = LinearRegression().fit(X, y)

print("Testing scenario(s):", scenarios)

# --- NEW: Calculate and print the R-squared value ---
r_squared = reg.score(X, y)
print(f"R-squared: {r_squared:.4f}")
# --- END NEW ---

# The coefficients are your new, data-driven weights!
metric_weights = reg.coef_

print("\n--- Data-Driven Weights for the Metric ---")
for factor, weight in zip(df_theta.columns, metric_weights):
    print(f"{factor}: {weight:.4f}")

# Now, you can create your final, math-backed metric
df_theta['Aggregated'] = np.dot(X, metric_weights)

# Rank the models by this new, objective metric
ranked_by_math_metric = df_theta.sort_values(by='Aggregated', ascending=False)
ranked_by_math_metric = ranked_by_math_metric[ranked_by_math_metric.index.str.contains("anthropic/cl|openai/gp|google/ge|meta/ll", case=False)]
df_theta.drop(columns=['Aggregated'], inplace=True)

print("\n--- Final Ranking based on Metric ---")
print(ranked_by_math_metric[['Aggregated']])

Testing scenario(s): ['med_qa']
R-squared: 0.5208

--- Data-Driven Weights for the Metric ---
F1: -0.1905
F2: 0.0352
F3: -0.0145
F4: 0.0553
F5: -0.0051
F6: 0.0152

--- Final Ranking based on Metric ---
                                               Aggregated
request.model                                            
openai/gpt-4o-2024-08-06                         0.412101
openai/gpt-4o-2024-05-13                         0.397825
meta/llama-3.2-90b-vision-instruct-turbo         0.376212
openai/gpt-4-turbo-2024-04-09                    0.364397
anthropic/claude-3-5-sonnet-20240620             0.342189
meta/llama-3.1-405b-instruct-turbo               0.340702
meta/llama-3.3-70b-instruct-turbo                0.334839
meta/llama-3.1-70b-instruct-turbo                0.328536
anthropic/claude-3-opus-20240229                 0.320255
google/gemini-1.5-flash-preview-0514             0.316960
openai/gpt-4-0613                                0.295258
google/gemini-1.5-pro-preview-0409          

In [4]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor   # <<-- changed
pd.set_option('display.max_rows', None)

# --- Assume you have these DataFrames ---
df_theta = ability_df
df_performance = pd.read_csv('../data/scenario_probs.csv')  # (models x scenarios) DataFrame of raw scores

# Step 1: Create the "Ground Truth" Score (your target variable 'y')
scenarios = ['med_qa']
ground_truth_scores = df_performance[scenarios].mean(axis=1)

# The factor scores are your predictor variables 'X'
X = df_theta
y = ground_truth_scores

# Step 2 & 3: Run the regression (XGBoost version)
reg = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
reg.fit(X, y)

print("Testing scenario(s):", scenarios)

# --- Calculate and print the R-squared value ---
r_squared = reg.score(X, y)
print(f"R-squared: {r_squared:.4f}")

# Instead of coefficients, we look at feature importances
importances = reg.feature_importances_

print("\n--- Data-Driven Feature Importances for the Metric ---")
for factor, importance in zip(df_theta.columns, importances):
    print(f"{factor}: {importance:.4f}")

# Now, you can create your final, math-backed metric
df_theta['Aggregated'] = reg.predict(X)

# Rank the models by this new, objective metric
ranked_by_math_metric = df_theta.sort_values(by='Aggregated', ascending=False)
ranked_by_math_metric = ranked_by_math_metric[ranked_by_math_metric.index.str.contains("anthropic/cl|openai/gp|google/ge|meta/ll", case=False)]
df_theta.drop(columns=['Aggregated'], inplace=True)

print("\n--- Final Ranking based on Metric ---")
print(ranked_by_math_metric[['Aggregated']])


Testing scenario(s): ['med_qa']
R-squared: 0.9997

--- Data-Driven Feature Importances for the Metric ---
F1: 0.3982
F2: 0.0647
F3: 0.1613
F4: 0.1642
F5: 0.1162
F6: 0.0954

--- Final Ranking based on Metric ---
                                               Aggregated
request.model                                            
openai/gpt-4o-2024-08-06                         0.871375
openai/gpt-4o-2024-05-13                         0.863433
meta/llama-3.1-405b-instruct-turbo               0.852641
anthropic/claude-3-5-sonnet-20241022             0.850028
anthropic/claude-3-5-sonnet-20240620             0.832373
meta/llama-3.2-90b-vision-instruct-turbo         0.825838
openai/gpt-4-0613                                0.820156
meta/llama-3.1-70b-instruct-turbo                0.818930
openai/gpt-4-1106-preview                        0.814443
meta/llama-3.3-70b-instruct-turbo                0.806820
openai/gpt-4-turbo-2024-04-09                    0.787015
anthropic/claude-3-opus-20240229   

In [7]:
# Compare regression predictions vs actual mean accuracy
print("=== Comparison: Regression Predictions vs Actual Mean Accuracy ===\n")

# Get regression predictions (using the XGBoost model from earlier)
regression_predictions = reg.predict(X)

# Get actual mean accuracy (from the current cell's calculation)
actual_mean = mean_accurate

# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Actual_Mean': actual_mean,
    'Regression_Pred': regression_predictions,
    'Difference': regression_predictions - actual_mean,
    'Abs_Difference': np.abs(regression_predictions - actual_mean)
})

# Round to 3 decimal places as requested
comparison_df = comparison_df.round(3)

# Sort by actual mean accuracy (descending) for easier comparison
comparison_df = comparison_df.sort_values('Actual_Mean', ascending=False)

print("Top 15 models comparison:")
print(comparison_df.head(15))

print(f"\nOverall Statistics:")
print(f"Mean Absolute Error: {comparison_df['Abs_Difference'].mean():.3f}")
print(f"Root Mean Square Error: {np.sqrt((comparison_df['Difference']**2).mean()):.3f}")
print(f"Correlation: {comparison_df['Actual_Mean'].corr(comparison_df['Regression_Pred']):.3f}")

# Show models with largest prediction errors
print(f"\nModels with largest prediction errors:")
worst_predictions = comparison_df.nlargest(5, 'Abs_Difference')[['Actual_Mean', 'Regression_Pred', 'Difference']]
print(worst_predictions)

=== Comparison: Regression Predictions vs Actual Mean Accuracy ===

Top 15 models comparison:
                                          Actual_Mean  Regression_Pred  \
request.model                                                            
openai/gpt-4o-2024-08-06                        0.871            0.871   
openai/gpt-4o-2024-05-13                        0.863            0.863   
anthropic/claude-3-5-sonnet-20241022            0.855            0.850   
meta/llama-3.1-405b-instruct-turbo              0.855            0.853   
anthropic/claude-3-5-sonnet-20240620            0.833            0.832   
meta/llama-3.2-90b-vision-instruct-turbo        0.826            0.826   
meta/llama-3.1-70b-instruct-turbo               0.822            0.819   
openai/gpt-4-1106-preview                       0.819            0.814   
openai/gpt-4-0613                               0.819            0.820   
deepseek-ai/deepseek-v3                         0.813            0.812   
meta/llama-3.3-70b